# TradaBoostR2

This notebook runs TradaBoostR2 as implemented in https://adapt-python.github.io/adapt/generated/adapt.utils.make_regression_da.html. 

Make sure to install adapt package (preferrably with Python 3.9), along with Tensorflow == 2.15. 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split

from adapt.instance_based import TrAdaBoostR2, TwoStageTrAdaBoostR2
from sklearn.metrics import mean_squared_error, mean_absolute_error

import itertools

from sklearn.preprocessing import StandardScaler

In [ ]:
seed_list = [1,2,3,4,5]
test_size_list = [0.7, 0.8, 0.9]

In [ ]:
predictor_columns = ['pzabovezmean', 'pzabove2', 'zq5', 'zq10',
    'zq15', 'zq20', 'zq25', 'zq30', 'zq35', 'zq40', 'zq45', 'zq50', 'zq55',
    'zq60', 'zq65', 'zq70', 'zq75', 'zq80', 'zq85', 'zq90', 'zq95',
    'zpcum1', 'zpcum2', 'zpcum3', 'zpcum4', 'zpcum5', 'zpcum6', 'zpcum7',
    'zpcum8', 'zpcum9'
    ]

In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense, Reshape
from tensorflow.keras.optimizers import Adam

def get_model():
    model = Sequential()
    model.add(Dense(12, activation='relu', input_shape=(30,)))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(12, activation='relu'))
    model.add(Dense(1))
    model.compile(optimizer=Adam(1e-3), loss='mean_squared_error')
    return model

In [ ]:
#ablation study for TradaBoostR2, Gaussian errors, with gaussian source domain errors
ablation_transfer_tradaboost_normal_normal = pd.DataFrame(columns = ['seed', 'splitting_variable', 'method',
                                   'n_estimators', 'lr', 'epochs', 'val_rmse', 'val_mae', 'rmse', 'mae'])

n_estimators_list = [5,20,35]
lr_list = [0.05, 0.1, 0.15]
epochs_list = [10, 30, 50]


# --- Step 2: Create full parameter grid ---
param_grid = list(itertools.product(
    n_estimators_list,
    lr_list,
    epochs_list
))

#ablation study for MLP
ablation_transfer_real = pd.DataFrame(columns = ['seed', 'target_column', 'target_instances', 'method', 'base_lr', 'fine_tuning_lr', 'dropout_rate', 'batch_norm', 'rmse', 'mae'])

test_size_list = [0.7, 0.8, 0.9] #0.8or 0.93
target_columns = ['Volume', 'Dgv']

fine_tuning_lrs = [1e-4, 5e-5]
base_lrs = [5e-4, 1e-4]
dropout_list = [0.0, 0.1]
include_batch_norm = [True, False]


# --- Step 3: Sample random combinations ---
#sampled_configs = random.sample(param_grid, n_samples)

for seed in seed_list:
    for test_size in test_size_list:
        for target_column in target_columns:

            #data from Svedala
            data_sweden = pd.read_csv(r'../datasets/rs_sweden.csv', index_col=[0])


            #evaluate and rain on latvia instead (keep naming for simplicity)
            #data from latvia target
            data_latvia = pd.read_csv(r'../datasets/rs_lettland.csv', index_col=[0])
            train_size = int((1-test_size)*len(data_latvia))
            data_latvia = data_latvia.rename(columns = {'H_AVERAGE': 'Hgv', 'D_AVERAGE': 'Dgv', 'VOLUME': 'Volume'})
            data_train, data_temp = train_test_split(data_latvia, test_size=test_size, random_state=seed)
            data_val, data_test = train_test_split(data_temp, test_size=0.5, random_state=seed)

            print(len(data_train), len(data_val), len(data_test))


            # --- Source domain ---
            X_source_train = np.array(data_sweden[predictor_columns], dtype=float)
            y_source_train = np.array(data_sweden[target_column])

            scaler_source = StandardScaler()
            X_source_train = scaler_source.fit_transform(X_source_train)


            # --- Target domain ---
            X_target_train = np.array(data_train[predictor_columns], dtype=float)
            y_target_train = np.array(data_train[target_column])

            X_target_val = np.array(data_val[predictor_columns], dtype=float)
            y_target_val = np.array(data_val[target_column])

            X_target_test = np.array(data_test[predictor_columns], dtype=float)
            y_target_test = np.array(data_test[target_column])

            # Fit scaler only on training data, then transform all target splits
            scaler_target = StandardScaler()
            X_target_train = scaler_target.fit_transform(X_target_train)
            X_target_val = scaler_target.transform(X_target_val)
            X_target_test = scaler_target.transform(X_target_test)

            for config in param_grid:
                n_estimators, lr, epochs = config


                method = f'TradaBoostR2'
                model = TrAdaBoostR2(get_model(),
                                n_estimators=n_estimators, lr=lr)

                model.fit(X_source_train, y_source_train, X_target_train, y_target_train, epochs = epochs, batch_size=32, verbose=0)
                preds = model.predict(X_target_test)
                val_preds = model.predict(X_target_val)
                val_rmse = np.sqrt(mean_squared_error(val_preds, y_target_val))
                val_mae = mean_absolute_error(val_preds, y_target_val)
                rmse = np.sqrt(mean_squared_error(preds, y_target_test))
                mae = mean_absolute_error(preds, y_target_test)
                ablation_transfer_tradaboost_normal_normal.loc[len(ablation_transfer_tradaboost_normal_normal)] = [seed, target_column, train_size, method, n_estimators,
                                                                                                                    lr, epochs, val_rmse, val_mae, rmse, mae]
                ablation_transfer_tradaboost_normal_normal.to_csv(f'results/tradaboost_ablation_housing.csv')

78 59 60


Iteration 0 - Error: 0.2084
Iteration 1 - Error: 0.2151
Iteration 2 - Error: 0.2220
Iteration 3 - Error: 0.2293
Iteration 4 - Error: 0.2303
Iteration 0 - Error: 0.2440
Iteration 1 - Error: 0.2408
Iteration 2 - Error: 0.2433
Iteration 3 - Error: 0.2571
Iteration 4 - Error: 0.2595
Iteration 0 - Error: 0.1658
Iteration 1 - Error: 0.1743
Iteration 2 - Error: 0.1768
Iteration 3 - Error: 0.1878
Iteration 4 - Error: 0.1914
Iteration 0 - Error: 0.2162
Iteration 1 - Error: 0.2271
Iteration 2 - Error: 0.2313
Iteration 3 - Error: 0.2393
Iteration 4 - Error: 0.2452
Iteration 0 - Error: 0.1819
Iteration 1 - Error: 0.1941
Iteration 2 - Error: 0.2004
Iteration 3 - Error: 0.2296
Iteration 4 - Error: 0.2181
Iteration 0 - Error: 0.2037
Iteration 1 - Error: 0.2027
Iteration 2 - Error: 0.2142
Iteration 3 - Error: 0.2335
Iteration 4 - Error: 0.2457
Iteration 0 - Error: 0.2047
Iteration 1 - Error: 0.2167
Iteration 2 - Error: 0.2308
Iteration 3 - Error: 0.2444
Iteration 4 - Error: 0.2627
Iteration